                # BootstrapFinetune (Apple Silicon / MPS)

                Bootstrap successful traces into training data, then update model weights rather than only prompt state.

                **Use it when:** You control a trainable model and want to distill accepted DSPy traces into a reusable local adapter.

                **What compilation changes:** A PEFT LoRA adapter for Qwen2.5-0.5B-Instruct; the prompt remains separately inspectable.

                Important configuration in this benchmark:

                - stock DSPy BootstrapFinetune with a Transformers/TRL provider boundary
- MPS selected on Apple Silicon; CPU remains an explicit fallback
- 18 full-profile training steps, batch size 1, LoRA rank 8, seed 42
- local self-teaching by default, so this row makes no OpenAI calls

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'bootstrap-finetune'
RUN_MODE = os.getenv("CHAPTER06_RUN", "inspect").lower()
print(f"mode={RUN_MODE!r}; optimizer={OPTIMIZER!r}")
print("safe default: inspect checked-in full-run artifacts; no API calls")

mode='inspect'; optimizer='bootstrap-finetune'
safe default: inspect checked-in full-run artifacts; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.BootstrapFinetune(
    metric=exact_match, train_kwargs=training_config,
    exclude_demos=True, num_threads=1,
)
optimized_detector = optimizer.compile(
    detector, trainset=trainset, teacher=local_teacher,
)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

BootstrapFinetune — frozen full-profile rerun
status: completed
task model: Qwen/Qwen2.5-0.5B-Instruct
test accuracy: 50.0%
uplift: +0.0 points vs its Qwen run baseline
optimization: $0.0000 and 1.1min
inference latency: mean 1.16s; p95 1.88s
reload checks: prompt=True; model=True; predictions=3/3 labels
comparison boundary: same frozen split, separate Qwen/MPS experiment; not Luna-comparable
complete run: chapter06/results/runs/full/bootstrap-finetune/20260715T034040.794581Z

Complete artifacts:
- complete_output: chapter06/results/runs/full/bootstrap-finetune/20260715T034040.794581Z/complete_output.txt
- lm_history: chapter06/results/runs/full/bootstrap-finetune/20260715T034040.794581Z/lm_history.jsonl
- metrics: chapter06/results/runs/full/bootstrap-finetune/20260715T034040.794581Z/metrics.json
- optimizer_state: chapter06/results/runs/full/bootstrap-finetune/20260715T034040.794581Z/optimizer_state.json
- predictions: chapter06/results/runs/full/bootstrap-finetune/20260715T034040.79

## Read the result

Read score and adapter evidence together. This Qwen/MPS result uses the same frozen split but is not an absolute Luna-to-Luna comparison; follow the training path for device, loss, and adapter metadata.

The next cell shows a bounded readable preview. The complete, lossless prompt and
serialized program remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 0

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Run it yourself (explicit opt-in)

The default `inspect` mode is offline and free. For a live run, first install from
the repository root with `python -m pip install -r requirements.txt`, configure
`OPENAI_API_KEY`, and restart the kernel.

- Bounded code-path check: launch Jupyter with `CHAPTER06_RUN=smoke`.
- Complete frozen-split rerun: launch Jupyter with `CHAPTER06_RUN=full`.

A full run is intentionally not triggered by opening or choosing “Run All”: it can
take minutes or incur model charges. The weight optimizers download and train a
small Qwen model locally through MPS/CPU and require the optional training stack. Live
artifacts are written to `chapter06/results/runs/<profile>/<optimizer>/<run-id>/`.
Rebuild the comparison afterward with `python -m chapter06.summarize_optimizer_results`.

In [4]:
if RUN_MODE == "inspect":
    print("Live run skipped. Set CHAPTER06_RUN=smoke or CHAPTER06_RUN=full explicitly.")
elif RUN_MODE in {"smoke", "full"}:
    from chapter06.optimizer_experiment import run_experiment

    live_result = run_experiment(OPTIMIZER, profile_name=RUN_MODE)
    print(live_result)
else:
    raise ValueError("CHAPTER06_RUN must be inspect, smoke, or full")

Live run skipped. Set CHAPTER06_RUN=smoke or CHAPTER06_RUN=full explicitly.
